In [ ]:
import numpy as np # linear algebra
import pandas as pd
import os
from functools import partial
import mimetypes
from pathlib import Path
from PIL import Image
import random
import matplotlib.pyplot as plt
import cv2

In [ ]:
import tensorflow as tf
from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import RMSprop,Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
# from keras.layers.normalization.batch_normalization import BatchNormalization
from tensorflow.keras.layers import Dense, Flatten, Conv2D
from tensorflow.keras.losses import sparse_categorical_crossentropy

In [ ]:
!wget https://github.com/narain1/projects/releases/download/v0.0.3/Downloads.zip
!unzip Downloads.zip -d images/

In [ ]:
img_fs = []
for p,d,fs in os.walk('/kaggle/input/fruits01/training'):
    for f in fs:
        img_fs.append(os.path.join(p, f))

In [ ]:
train_datagen = ImageDataGenerator(
        rescale=1 / 255.0,
        rotation_range=20,
        zoom_range=0.10,
        width_shift_range=0.05,
        height_shift_range=0.05,
        shear_range=0.05,
        horizontal_flip=True,
        fill_mode="nearest",
        validation_split=0.20)

test_datagen = ImageDataGenerator(rescale=1 / 255.0)

In [ ]:
data = train_datagen.flow_from_directory(
    directory='fruits01/training/training',
    target_size=(128, 128),
    color_mode='rgb',
    classes=None,
    class_mode='categorical',
    batch_size=64,
    shuffle=True,
    interpolation='nearest',
    keep_aspect_ratio=False,
)

In [ ]:
data_test = train_datagen.flow_from_directory(
    directory='fruits01/testing/testing',
    target_size=(128, 128),
    color_mode='rgb',
    classes=None,
    class_mode='categorical',
    batch_size=64,
    shuffle=True,
    interpolation='nearest',
    keep_aspect_ratio=False,
)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(10, 10))

for k, ax in enumerate(axes.ravel()):
    img = Image.open(random.choice(img_fs)).convert('RGB')
#     pixels = list(img.getdata())
#     img = convert_to_rgb565(random.choice(fs))
    ax.imshow(img)
    ax.axis('off')
    
fig.savefig('illustration.png')

In [ ]:
data.class_indices

In [ ]:
# resize_and_rescale = tf.keras.Sequential([
#   layers.experimental.preprocessing.Resizing(256, 256),
#   layers.experimental.preprocessing.Rescaling(1./255),
# ])

In [ ]:
# data_train.class_indices

In [ ]:
model = Sequential([
#     resize_and_rescale,
    layers.Conv2D(32, kernel_size = (3,3), activation='relu', input_shape=(128,128,3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, kernel_size = (3,3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, kernel_size =(3,3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, kernel_size =(3,3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, kernel_size = (3,3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.25),
    layers.Dense(3, activation='softmax'),
])

# model.build(input_shape=input_shape)

In [ ]:
model.summary()

In [ ]:
model.compile(loss="categorical_crossentropy",optimizer="adam",metrics=['accuracy'])

In [ ]:
model.fit(data,
        validation_data = data,
        steps_per_epoch = data.n//data.batch_size,
        validation_steps = data.n//data.batch_size,
        epochs=5)

In [ ]:
output = model.predict(data_test)

In [ ]:
data_test.classes

In [ ]:
(np.argmax(output, axis=1) == data_test.classes).mean()

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

In [ ]:
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)